# 具有反馈循环的 RAG 系统：提高检索和响应质量

## 概述

该系统实现了带有集成反馈回路的检索增强生成（RAG）方法。它旨在通过纳入用户反馈并动态调整检索过程来提高响应的质量和相关性。

## 动机

由于检索过程或底层知识库的限制，传统 RAG 系统有时会产生不一致或不相关的响应。通过实施反馈循环，我们可以：

1. 不断提高检索文献质量
2. 增强生成响应的相关性
3. 随着时间的推移，使系统适应用户的偏好和需求

## 关键组件

1. **PDF内容提取**：从PDF文档中提取文本
2. **Vectorstore**：存储和索引文档嵌入以实现高效检索
3. **搜索器**：根据用户查询获取相关文档
4. **语言模型**：使用检索到的文档生成响应
5. **反馈收集**：收集用户对响应质量和相关性的反馈
6. **反馈存储**：保留用户反馈以供将来使用
7. **相关性分数调整**：根据反馈修改文档相关性
8. **索引微调**：使用累积的反馈定期更新向量库

## 方法详细信息

### 1. 初始设置
- 系统读取PDF内容并创建向量存储
- 使用向量存储初始化检索器
- 为响应生成建立语言模型（LLM）

### 2. 查询处理
- 当用户提交查询时，搜索器会获取相关文档
- 法学硕士根据检索到的文档生成响应

### 3.反馈收集
- 系统收集用户对响应的相关性和质量的反馈
- 反馈存储在 JSON 文件中以实现持久性

### 4.相关性分数调整
- 对于后续查询，系统加载之前的反馈
- 法学硕士评估过去反馈与当前查询的相关性
- 文档相关性分数根据此评估进行调整

### 5.搜索器更新
- 搜索器根据调整后的文档分数进行更新
- 这确保未来的检索受益于过去的反馈

### 6.定期索引微调
- 系统定期微调索引
- 高质量的反馈用于创建附加文档
- 向量存储库已使用这些新文档进行更新，提高了整体检索质量

## 这种方法的好处

1. **持续改进**：系统从每次交互中学习，逐渐增强其性能。
2. **个性化**：通过整合用户反馈，系统可以随着时间的推移适应个人或群体的偏好。
3. **增加相关性**：反馈循环有助于在将来的检索中优先考虑更相关的文档。
4. **质量控制**：随着系统的发展，低质量或不相关的响应不太可能重复。
5. **适应性**：系统可以根据用户需求或文档内容随时间的变化进行调整。

## 结论

这种具有反馈环路的 RAG 系统比传统 RAG 实现取得了重大进步。通过不断地从用户交互中学习，它提供了一种更加动态、自适应和以用户为中心的信息检索和响应生成方法。该系统在信息准确性和相关性至关重要以及用户需求可能随时间变化的领域中特别有价值。

虽然与基本 RAG 系统相比，该实施增加了复杂性，但响应质量和用户满意度方面的优势使其对于需要高质量、上下文感知信息检索和生成的应用程序来说是值得的投资。

<div style="text-align: center;">

<img src="../images/retrieval_with_feedback_loop.svg" alt="retrieval with feedback loop" style="width:40%; height:auto;">
</div>

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install langchain langchain-openai python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [ ]:
import os
import sys
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
import json
from typing import List, Dict, Any


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

### 定义文档路径

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/feedback_data.json https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/feedback_data.json
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [2]:
path = "data/Understanding_Climate_Change.pdf"

### 创建管理存储和检索 QA 链

In [3]:
content = read_pdf_to_string(path)
vectorstore = encode_from_string(content)
retriever = vectorstore.as_retriever()

llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=4000)
qa_chain = RetrievalQA.from_chain_type(llm, retriever=retriever)

### 在字典中格式化用户反馈的函数

In [4]:
def get_user_feedback(query, response, relevance, quality, comments=""):
    return {
        "query": query,
        "response": response,
        "relevance": int(relevance),
        "quality": int(quality),
        "comments": comments
    }

### 将反馈存储在 json 文件中的函数

In [5]:
def store_feedback(feedback):
    with open("data/feedback_data.json", "a") as f:
        json.dump(feedback, f)
        f.write("\n")

### 读取反馈文件的函数

In [6]:
def load_feedback_data():
    feedback_data = []
    try:
        with open("data/feedback_data.json", "r") as f:
            for line in f:
                feedback_data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print("No feedback data file found. Starting with empty feedback.")
    return feedback_data

### 根据反馈文件调整文件相关性的功能

In [7]:
class Response(BaseModel):
    answer: str = Field(..., title="The answer to the question. The options can be only 'Yes' or 'No'")

def adjust_relevance_scores(query: str, docs: List[Any], feedback_data: List[Dict[str, Any]]) -> List[Any]:
    # Create a prompt template for relevance checking
    relevance_prompt = PromptTemplate(
        input_variables=["query", "feedback_query", "doc_content", "feedback_response"],
        template="""
        Determine if the following feedback response is relevant to the current query and document content.
        You are also provided with the Feedback original query that was used to generate the feedback response.
        Current query: {query}
        Feedback query: {feedback_query}
        Document content: {doc_content}
        Feedback response: {feedback_response}
        
        Is this feedback relevant? Respond with only 'Yes' or 'No'.
        """
    )
    llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=4000)

    # Create an LLMChain for relevance checking
    relevance_chain = relevance_prompt | llm.with_structured_output(Response)

    for doc in docs:
        relevant_feedback = []
        
        for feedback in feedback_data:
            # Use LLM to check relevance
            input_data = {
                "query": query,
                "feedback_query": feedback['query'],
                "doc_content": doc.page_content[:1000],
                "feedback_response": feedback['response']
            }
            result = relevance_chain.invoke(input_data).answer
            
            if result == 'yes':
                relevant_feedback.append(feedback)
        
        # Adjust the relevance score based on feedback
        if relevant_feedback:
            avg_relevance = sum(f['relevance'] for f in relevant_feedback) / len(relevant_feedback)
            doc.metadata['relevance_score'] *= (avg_relevance / 3)  # Assuming a 1-5 scale, 3 is neutral
    
    # Re-rank documents based on adjusted scores
    return sorted(docs, key=lambda x: x.metadata['relevance_score'], reverse=True)

### 微调向量索引的函数，以包含收到良好反馈的查询+答案

In [13]:
def fine_tune_index(feedback_data: List[Dict[str, Any]], texts: List[str]) -> Any:
    # Filter high-quality responses
    good_responses = [f for f in feedback_data if f['relevance'] >= 4 and f['quality'] >= 4]
    
    # Extract queries and responses, and create new documents
    additional_texts = []
    for f in good_responses:
        combined_text = f['query'] + " " + f['response']
        additional_texts.append(combined_text)

    # make the list a string
    additional_texts = " ".join(additional_texts)
    
    # Create a new index with original and high-quality texts
    all_texts = texts + additional_texts
    new_vectorstore = encode_from_string(all_texts)
    
    return new_vectorstore

### 演示如何检索有关用户反馈的答案

In [29]:

query = "What is the greenhouse effect?"

# Get response from RAG system
response = qa_chain(query)["result"]

relevance = 5
quality = 5

# Collect feedback
feedback = get_user_feedback(query, response, relevance, quality)

# Store feedback
store_feedback(feedback)

# Adjust relevance scores for future retrievals
docs = retriever.get_relevant_documents(query)
adjusted_docs = adjust_relevance_scores(query, docs, load_feedback_data())

# Update the retriever with adjusted docs
retriever.search_kwargs['k'] = len(adjusted_docs)
retriever.search_kwargs['docs'] = adjusted_docs

### 定期微调向量存储

In [14]:
# Periodically (e.g., daily or weekly), fine-tune the index
new_vectorstore = fine_tune_index(load_feedback_data(), content)
retriever = new_vectorstore.as_retriever()

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--retrieval-with-feedback-loop)